# Modeling

In [7]:
import os, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, make_scorer, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.calibration import CalibratedClassifierCV

from xgboost import XGBClassifier

In [2]:
PROCESSED_DIR = r"D:\Ameng\Data Science Project\heart-failure-prediction\data\processed"
X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze("columns")

In [3]:
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Target train:", y_train.value_counts(normalize=True))

X_train: (239, 12) X_test: (60, 12)
Target train: DEATH_EVENT
0    0.677824
1    0.322176
Name: proportion, dtype: float64


In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorers = {
    "AUC": make_scorer(roc_auc_score, response_method="predict_proba"),
    "F1": make_scorer(f1_score),
    "PRE": make_scorer(precision_score),
    "REC": make_scorer(recall_score),
}

def cv_table(X, y, models):
    rows = []
    for name, model in models.items():
        r = {m: cross_val_score(model, X, y, cv=cv, scoring=s).mean().round(4)
             for m, s in scorers.items()}
        r |= {"model": name}; rows.append(r)
    return pd.DataFrame(rows).set_index("model")

In [8]:
models = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "SVM": CalibratedClassifierCV(SVC(kernel="rbf", class_weight="balanced"), ensemble=False),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
}
result = cv_table(X_train, y_train, models)
print(result.sort_values("AUC", ascending=False))

                     AUC      F1     PRE     REC
model                                           
XGBoost           0.9109  0.7246  0.8013  0.6750
RandomForest      0.9068  0.7443  0.7424  0.7550
LogReg            0.8984  0.7663  0.7362  0.8042
GradientBoosting  0.8964  0.7036  0.7616  0.6642
SVM               0.8844  0.7371  0.8155  0.6758
kNN               0.8206  0.5730  0.8078  0.4583


Based on 5-fold cross-validation result, Logistic Regression chosen as the best model because it has the highest F1-score (0.7663) and recall (0.8042). <br>
Although XGBoost achieved the highest AUC (0.9109), Logistic regression performs better at detecting death cases, which is a crucial consideration in this instance.

In [11]:
param_lr = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0, 1]
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear"),
    param_grid=param_lr,
    cv=cv, scoring="roc_auc", n_jobs=-1)
grid_lr.fit(X_train, y_train)
print("Best param:", grid_lr.best_params_)
print("Best CV AUC:", grid_lr.best_score_.round(4))

Best param: {'C': 100, 'l1_ratio': 0}
Best CV AUC: 0.9038


In [13]:
final_models = {
    "LogReg": grid_lr.best_estimator_,
    "RandomForest": models["RandomForest"],
    "SVM": models["SVM"],
    "kNN": models["kNN"],
    "GradientBoosting": models["GradientBoosting"],
    "XGBoost": models["XGBoost"]
}
res_final = cv_table(X_train, y_train, final_models)
print(res_final.sort_values("AUC", ascending=False))

                     AUC      F1     PRE     REC
model                                           
XGBoost           0.9109  0.7246  0.8013  0.6750
RandomForest      0.9068  0.7443  0.7424  0.7550
LogReg            0.9038  0.7706  0.7432  0.8042
GradientBoosting  0.8964  0.7036  0.7616  0.6642
SVM               0.8844  0.7371  0.8155  0.6758
kNN               0.8206  0.5730  0.8078  0.4583
